# OUSIA Gemma Phase 0 — Foundation Training
**Gemma 4 Good Hackathon** | Colab Pro (A100/H100)

Trains google/gemma-4-E4B-it on OpenHermes-2.5 + OASST1 to build a strong agentic foundation before Phase 1 (anti-sycophancy + emotional regulation).

**This is the foundation.** Phase 1 (OUSIA-Gemma.ipynb) builds on top of this adapter.

Base: google/gemma-4-E4B-it (clean, no prior adapter)  
Datasets: OpenHermes-2.5 (20K) + OASST1 English (15K)  
LoRA: r=16, seq=1024, batch=4, grad_accum=4 → effective batch 16  
Learning rate: 2e-4 (clean foundation, no existing adapter to preserve)  
Est. runtime: 2–3 hours on A100

## 1. Setup

In [ ]:
# Upgrade transformers first (required for Gemma 4)
!pip install -q --upgrade transformers huggingface_hub
!pip install -q transformers peft datasets accelerate bitsandbytes torch trl
!pip install -q tensorboard

## 2. Clone Repo

In [ ]:
# Clone training repo
!git clone https://github.com/plntrprotocol/aureth-training.git /content/aureth-training 2>/dev/null || echo "already mounted"
%cd /content/aureth-training

# HF token from Colab Secrets
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print(f"HF_TOKEN set ({len(hf_token)} chars)")
else:
    raise RuntimeError("HF_TOKEN not found. Add via ⚙️ → Secrets.")

## 3. Load Gemma Base + Tokenizer

**CRITICAL:** Gemma4 requires `attn_implementation="eager"` for QLoRA compatibility. sdpa/flash_attention_2 have issues with 4-bit quantization.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "google/gemma-4-E4B-it"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {tokenizer.vocab_size} tokens")

print(f"\nLoading Gemma-4-E4B-it (QLoRA 4-bit, eager attention)...")
# attn_implementation="eager" is CRITICAL for Gemma4 + QLoRA
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    device_map="cuda",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
print(f"Model loaded: {model.num_parameters():,} parameters")

## 4. Replace Gemma4ClippableLinear → Linear (PEFT Requirement)

Gemma4ClippableAttention wraps q/k/v in Gemma4ClippableLinear. PEFT cannot inject LoRA into these. Replace with standard nn.Linear before applying LoRA.

In [ ]:
import torch.nn as nn

# Get the model base
if hasattr(model, 'language_model'):
    model_base = model.language_model
else:
    model_base = model

# Replace Gemma4ClippableLinear → standard Linear
try:
    from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
    
    replaced = 0
    for name, module in list(model_base.named_modules()):
        if isinstance(module, Gemma4ClippableLinear):
            parent_name, attr_name = name.rsplit('.', 1)
            parent = model_base.get_submodule(parent_name)
            
            new_linear = nn.Linear(
                module.linear.in_features,
                module.linear.out_features,
                bias=module.linear.bias is not None
            )
            new_linear.weight = module.linear.weight
            new_linear.bias = module.linear.bias
            setattr(parent, attr_name, new_linear)
            replaced += 1
    
    print(f"Replaced {replaced} Gemma4ClippableLinear → Linear")
    if replaced == 0:
        print("No ClippableLinear found — may already be using eager attention correctly.")
        
except ImportError as e:
    print(f"Gemma4ClippableLinear not in this transformers version: {e}")
    print("Proceeding — eager attention should handle this.")

## 5. Apply LoRA Adapter

Full layer targeting for maximum foundation capacity. This is a CLEAN foundation — no existing adapter to preserve.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Full attention + MLP layers
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("Applying LoRA adapter (clean foundation)...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Load Foundation Datasets

OpenHermes-2.5 (high-quality instruction-tuned conversations) + OASST1 English (diverse dialogue).  
These build the agentic foundation before anti-sycophancy training.

In [ ]:
from datasets import load_dataset

MAX_SEQ = 1024

print("Loading OpenHermes-2.5 (streaming, 20K examples)...")
openhermes = load_dataset(
    "mistralai/OpenHermes-2.5",
    split="train",
    streaming=True
)
print(f"OpenHermes-2.5: {openhermes.num_rows if hasattr(openhermes,'num_rows') and openhermes.num_rows else 'streaming' } examples")

print("\nLoading OASST1 English subset (streaming, 15K examples)...")
oasst = load_dataset(
    "OpenAssistant/oasst1",
    subset="lang_en",
    split="train",
    streaming=True
)
print(f"OASST1: {oasst.num_rows if hasattr(oasst,'num_rows') and oasst.num_rows else 'streaming'} examples")

## 7. Format + Tokenize

Format conversations to Gemma-compatible structure and tokenize.

In [ ]:
import itertools

def extract_text_from_example(example):
    """Extract text from various dataset formats."""
    # OpenHermes-2.5 format
    if "text" in example:
        return example["text"]
    # OASST1 format with conversations
    if "conversations" in example:
        parts = []
        for msg in example["conversations"]:
            role = msg.get("role", "user")
            if role == "assistant":
                role = "assistant"
            elif role == "prompter":
                role = "user"
            parts.append(f"<|{role}|>\n{msg.get('content', '')}")
        return "<|end|>".join(parts) + "<|end|>"
    # Fallback
    return str(example)

def tokenize_fn(example):
    text = extract_text_from_example(example)
    if not text or len(text.strip()) < 20:
        return {"input_ids": [], "labels": []}
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Take samples (streaming)
print("Sampling OpenHermes-2.5 (20K)...")
oh_sample = openhermes.take(20000)
oh_tokenized = oh_sample.map(
    tokenize_fn,
    remove_columns=oh_sample.column_names,
    batched=False
)
# Filter empty
oh_tokenized = oh_tokenized.filter(lambda x: len(x.get("input_ids", [])) > 0)
print(f"OpenHermes tokenized: {len(oh_tokenized)} examples")

print("\nSampling OASST1 English (15K)...")
oasst_sample = oasst.take(15000)
oasst_tokenized = oasst_sample.map(
    tokenize_fn,
    remove_columns=oasst_sample.column_names,
    batched=False
)
oasst_tokenized = oasst_tokenized.filter(lambda x: len(x.get("input_ids", [])) > 0)
print(f"OASST1 tokenized: {len(oasst_tokenized)} examples")

## 8. Interleave + Combine Datasets

Mix OpenHermes (55%) + OASST1 (45%) with weighted interleaving for diverse foundation.

In [ ]:
from datasets import interleave_datasets

# Interleave: 55% OpenHermes, 45% OASST1
print("Interleaving datasets...")
combined = interleave_datasets(
    [oh_tokenized, oasst_tokenized],
    weights=[0.55, 0.45],
    stopping_strategy="all_datasts_exhausted"
)
print(f"Combined dataset: {len(combined)} examples")
print(f"Source mix: ~11K OpenHermes + ~6.8K OASST1")

## 9. Train Foundation

Standard LR (2e-4) for clean foundation. Gradient checkpointing for memory efficiency. bf16 on A100.

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
import os

OUTPUT_DIR = "/content/output/ousia-gemma-phase0"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch = 16
    num_train_epochs=2,
    learning_rate=2e-4,               # standard LR — clean foundation
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.5,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    bf16=True,                        # A100 supports bfloat16
    optim="paged_adamw_8bit",
    dataloader_num_workers=4,
    report_to=["tensorboard"],
    remove_unused_columns=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=MAX_SEQ,
    label_pad_token_id=tokenizer.pad_token_id,
)

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=combined,
    data_collator=data_collator,
    max_seq_length=MAX_SEQ,
    tokenizer=tokenizer,
    dataset_text_field="text",
)

print("\nFoundation Training Config:")
print(f"  Model: {MODEL_ID}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Dataset: {len(combined)} examples (~11K Hermes + ~6.8K OASST1)")
print(f"  LoRA: r=16, alpha=32, target={len(TARGET_MODULES)} modules")
print(f"  bf16: True (A100)")
print("\nStarting foundation training... (2–3 hours on A100)")
trainer.train()

## 10. Save Phase 0 Adapter

Download this zip — it becomes the base for OUSIA-Gemma.ipynb (Phase 1).

In [ ]:
# Save Phase 0 adapter
adapter_path = "/content/ou sia-gemma-phase0-adapter"
trainer.save_model(adapter_path)
print(f"Phase 0 adapter saved: {adapter_path}")

# Create zip for download
!cd /content && zip -r ousia-gemma-phase0-adapter.zip ousia-gemma-phase0-adapter/
print("\n📦 Download: ousia-gemma-phase0-adapter.zip")
print("   → Upload this to OUSIA-Gemma.ipynb (Phase 1)")

## 11. Quick Validation — Chat Test

Test the Phase 0 adapter for basic instruction-following quality.

In [ ]:
def chat_test(prompt, max_new_tokens=200):
    inputs = tokenizer(f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n", return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("<|assistant|>")[-1].strip()

print("Phase 0 Chat Tests")
print("=" * 50)
for prompt in [
    "Explain quantum entanglement in one sentence.",
    "Write a Python function to reverse a linked list.",
    "What are the main themes of Hamlet?",
]:
    print(f"\nQ: {prompt}")
    print(f"A: {chat_test(prompt)}")